In [18]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv")
df["TotalCharges"] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df_cleaned = df.dropna(subset=['TotalCharges']).copy()
X = df_cleaned.drop(columns=['Partner', 'customerID', 'Churn'])
y = df_cleaned['Partner'] # Target variable: Partner (categorical 'Yes'/'No')

print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features")
print(f"Class distribution: {y.value_counts(normalize=True).to_dict()}")

# ============================================================
# STEP 1: SPLIT FIRST! (Before ANY preprocessing)
# ============================================================
# Use stratify for classification to maintain class proportions

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)
print(f"\nTrain: {len(X_train)} | Test: {len(X_test)}")
print(f"\nTrain class balance: {y_train.value_counts(normalize=True).to_dict()}")
print(f"\nTest class balance: {y_test.value_counts(normalize=True).to_dict()}")

numeric_cols = X_train.select_dtypes(include=np.number).columns
categorical_cols = X_train.select_dtypes(include=['category','object']).columns

preprocessor = ColumnTransformer(
  transformers=[
    ('num', Pipeline([
      ('imputer', SimpleImputer(strategy='median')),
      ('scaler', StandardScaler())
    ]), numeric_cols),
    ('cat', Pipeline([
      ('imputer', SimpleImputer(strategy='most_frequent')),
      ('scaler', OneHotEncoder(handle_unknown='ignore'))
    ]),categorical_cols)
  ]
)

# ============================================================
# STEP 2: BUILD PIPELINE (preprocessing + model)
# ============================================================
# Pipeline ensures preprocessing is fit ONLY on training fold

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# ============================================================
# STEP 3: CROSS-VALIDATION ON TRAIN SET
# ============================================================
# StratifiedKFold maintains class proportions in each fold


cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
cv_scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='accuracy')
print(f"\n 5 Folds CV Results: ")
print(f"\nScores: {[f'{s:.3f}' for s in cv_scores]}")
print(f"\n Mean: {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}")


# ============================================================
# STEP 4: FINAL EVALUATION (Test set used ONCE)
# ============================================================

pipeline.fit(X_train,y_train)

test_score = pipeline.score(X_test, y_test)
print(f"\nFinal Test Score: {test_score:.3f}")
print("(Should be close to CV score if no leakage)")

Dataset: 7032 samples, 18 features
Class distribution: {'No': 0.5174914675767918, 'Yes': 0.4825085324232082}

Train: 5625 | Test: 1407

Train class balance: {'No': 0.5175111111111111, 'Yes': 0.4824888888888889}

Test class balance: {'No': 0.5174129353233831, 'Yes': 0.48258706467661694}

 5 Folds CV Results: 

Scores: ['0.718', '0.716', '0.739', '0.721', '0.762']

 Mean: 0.731 +/- 0.017

Final Test Score: 0.701
(Should be close to CV score if no leakage)


Special Cases: Time Series and Grouped Data

Correct splitting for temporal and hierarchical data structures

In [26]:
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit, GroupKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor


# ============================================================
# TIME SERIES DATA: Stock prices, weather, sales forecasting
# NEVER randomly shuffle time series!
# ============================================================

dates = pd.date_range('2020-01-01', periods=1000, freq='D')
df_ts = pd.DataFrame({
    'date': dates,
    'feature1': np.random.randn(1000).cumsum(),
    'feature2': np.random.randn(1000),
    'target': np.random.randn(1000).cumsum()
})

X_ts = df_ts[['feature1','feature2']].values
y_ts = df_ts['target'].values

time_series_split = TimeSeriesSplit(n_splits=5,gap=7) # 7 day gap to prevent leakage

print("Time Series Cross-Validation:")
for i, (train_idx, test_idx) in enumerate(time_series_split.split(X_ts)):
    train_end = df_ts.iloc[train_idx[-1]]['date']
    test_start = df_ts.iloc[test_idx[0]]['date']
    print(f"  Fold {i+1}: Train ends {train_end.date()} | "
          f"Test starts {test_start.date()}")

# ============================================================
# GROUPED DATA: Multiple samples per user/patient/entity
# Keep all samples from same group in same fold
# ============================================================

df_grouped = pd.DataFrame({
    'user_id': np.repeat(range(100), 10),  # 100 users, 10 each
    'feature1': np.random.randn(1000),
    'feature2': np.random.randn(1000),
    'target': np.random.randint(0, 2, 1000)
})

X_grouped = df_grouped[['feature1', 'feature2']].values
y_grouped = df_grouped['target'].values
groups = df_grouped['user_id'].values

gkfold = GroupKFold(n_splits=5)
print("\nGroup K-Fold Cross-Validation:")
for i, (train_idx, test_idx) in enumerate(gkfold.split(
    X_grouped, y_grouped, groups
)):
    train_users = np.unique(groups[train_idx])
    test_users = np.unique(groups[test_idx])
    overlap = set(train_users) & set(test_users)
    print(f"  Fold {i+1}: {len(train_users)} train users | "
          f"{len(test_users)} test users | Overlap: {len(overlap)}")

# Compare honest vs leaked scores
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', GradientBoostingRegressor(n_estimators=50, random_state=42))
])

group_scores = cross_val_score(
    pipeline, X_grouped, y_grouped, cv=gkfold, groups=groups
)
print(f"\nGroupKFold Score: {group_scores.mean():.3f} "
      f"(+/- {group_scores.std()*2:.3f})")


Time Series Cross-Validation:
  Fold 1: Train ends 2020-06-11 | Test starts 2020-06-19
  Fold 2: Train ends 2020-11-24 | Test starts 2020-12-02
  Fold 3: Train ends 2021-05-09 | Test starts 2021-05-17
  Fold 4: Train ends 2021-10-22 | Test starts 2021-10-30
  Fold 5: Train ends 2022-04-06 | Test starts 2022-04-14

Group K-Fold Cross-Validation:
  Fold 1: 80 train users | 20 test users | Overlap: 0
  Fold 2: 80 train users | 20 test users | Overlap: 0
  Fold 3: 80 train users | 20 test users | Overlap: 0
  Fold 4: 80 train users | 20 test users | Overlap: 0
  Fold 5: 80 train users | 20 test users | Overlap: 0

GroupKFold Score: -0.054 (+/- 0.071)


Data Leakage Detection and Prevention

Demonstrating leakage vs correct pipeline with measurable impact

In [27]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# ============================================================
# CREATE DATASET WITH KNOWN PROPERTIES
# ============================================================
X, y = make_classification(
    n_samples=1000, n_features=20,
    n_informative=5, random_state=42
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ============================================================
# WRONG: Preprocessing BEFORE cross-validation (LEAKAGE!)
# ============================================================
# Scaler sees ALL data including future test folds
scaler = StandardScaler()
X_leaked = scaler.fit_transform(X)  # LEAKAGE: test info in scaler

leaked_scores = cross_val_score(
    LogisticRegression(random_state=42),
    X_leaked, y, cv=cv
)
print(f"WITH leakage:    {leaked_scores.mean():.4f} "
      f"(+/- {leaked_scores.std()*2:.4f})")

# ============================================================
# RIGHT: Pipeline handles preprocessing per fold
# ============================================================
pipeline = Pipeline([
    ('scaler', StandardScaler()),   # Refitted each fold
    ('model', LogisticRegression(random_state=42))
])

clean_scores = cross_val_score(pipeline, X, y, cv=cv)
print(f"WITHOUT leakage: {clean_scores.mean():.4f} "
      f"(+/- {clean_scores.std()*2:.4f})")

# ============================================================
# MEASURE THE LEAKAGE IMPACT
# ============================================================
gap = leaked_scores.mean() - clean_scores.mean()
print(f"\nLeakage inflated score by: {gap*100:.2f}%")
print("In real datasets with more features, this gap can be 5-15%!")

# ============================================================
# BEST PRACTICE: Full pipeline with multiple preprocessors
# ============================================================
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif

full_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('selector', SelectKBest(f_classif, k=10)),  # Also needs fit!
    ('model', LogisticRegression(random_state=42))
])

full_scores = cross_val_score(full_pipeline, X, y, cv=cv)
print(f"\nFull pipeline:   {full_scores.mean():.4f} "
      f"(+/- {full_scores.std()*2:.4f})")
print("Pipeline ensures imputer, scaler, AND selector fit per fold")

WITH leakage:    0.8120 (+/- 0.0377)
WITHOUT leakage: 0.8120 (+/- 0.0377)

Leakage inflated score by: 0.00%
In real datasets with more features, this gap can be 5-15%!

Full pipeline:   0.8150 (+/- 0.0400)
Pipeline ensures imputer, scaler, AND selector fit per fold
